# ¿Vale la pena un Vision Transformer para clasificar residuos?
## Comparación de redes convolucionales contra un Vision Transformer, pensando en un sistema embebido

**Gabriela Medina Navidad**
Proyecto final de Inteligencia Artificial (DCI019), Doctorado en Ciencias e Ingeniería, Universidad Autónoma de Guadalajara.
Profesor: Dr. Eduardo Ulises Moya Sánchez.
Código: [github.com/gnavidad1978/clasificacion_residuos_ia](https://github.com/gnavidad1978/clasificacion_residuos_ia)

## 1. El problema

Mi tesis doctoral trata de un bote de basura inteligente para el campus. Cada bote tiene una cámara: la persona acerca su residuo, el sistema reconoce de qué material es y le indica si ese es el bote correcto **antes** de abrir la compuerta. Así se evita que se mezclen los residuos y, de paso, la gente aprende a separar.

Para que funcione en el campus, el modelo tiene que correr dentro del propio bote, en una computadora pequeña como una Raspberry Pi, alimentada por un panel solar. Eso significa que no basta con que el modelo acierte: también tiene que ser **ligero y rápido**.

Las categorías no las inventé; salen de la norma de Jalisco para separar residuos, la **NAE-SEMADES-007/2008** [9]. Uso seis: vidrio, plástico, metal, papel y cartón, residuos inorgánicos de difícil reciclaje (el término exacto de la norma) y orgánico.

## 2. La pregunta que quise responder

Para clasificar imágenes hoy hay dos grandes familias de redes neuronales:

| Familia | Cómo "mira" una foto | Fama |
|---|---|---|
| **Red convolucional (CNN)** | Recorre la imagen con pequeñas lupas que detectan bordes, luego texturas, luego partes y al final objetos completos | La opción clásica, probada y eficiente |
| **Vision Transformer (ViT)** | Corta la foto en cuadritos, como un rompecabezas, y compara cada cuadrito con todos los demás para decidir qué es | Lo más moderno, pero pide más datos y más cómputo |

Siguiendo la recomendación del profesor, mi pregunta fue: **¿el Vision Transformer da un desempeño mejor que justifique su costo, o una CNN logra lo mismo con mucho menos cómputo?** Para un bote con panel solar, esa diferencia de costo importa mucho.

No entrené ninguna red desde cero. Usé **aprendizaje por transferencia**: tomé redes que ya habían aprendido a reconocer formas y texturas con más de un millón de fotos de la base ImageNet [7], les cambié la última capa para que respondieran con mis seis categorías y las volví a entrenar con fotos de residuos. Es como contratar a alguien que ya sabe ver y solo enseñarle el trabajo nuevo.

Comparé estos cuatro modelos:

| Modelo | Familia | Por qué lo elegí |
|---|---|---|
| MobileNetV3-Large [3] | CNN | Diseñada para teléfonos y equipos pequeños |
| ResNet-50 [2] | CNN | La referencia estándar, con conexiones residuales |
| YOLO11n-cls [5] | CNN | Es de la familia YOLO, la misma del detector que usaré en mi tesis |
| ViT-B/16 [4] | Transformer | El Vision Transformer base, para la comparación que pidió el profesor |

## 3. Cómo organicé el proyecto

Seguí la misma organización que vimos en clase: el código vive en un repositorio de GitHub con control de versiones, y este cuaderno solo ejecuta comandos. Cada experimento se configura en un archivo de texto YAML, así que para cambiar de modelo o de parámetros no toco el código.

| Archivo | Qué hace |
|---|---|
| `prepare_data.py` | Descarga las fotos, las acomoda en mis seis categorías y las separa en grupos |
| `model.py` | Contiene los cuatro modelos (una "fábrica de modelos") |
| `dataset.py` | Lee las fotos y las prepara para la red |
| `main.py` | Entrena con PyTorch Lightning [8] leyendo el YAML |
| `configs/*.yaml` | Un archivo de configuración por modelo |
| `benchmark.py` | Mide cuánto cómputo necesita cada modelo |
| `report.py` | Arma la tabla comparativa y las gráficas |

En esta primera celda defino qué modelos comparo y con qué semillas. Entrené cada modelo **tres veces** con semillas distintas (42, 7 y 2026), porque una sola corrida puede salir bien o mal por suerte; con tres puedo reportar el promedio y cuánto varía.

In [ ]:
import os
import glob
import json
from IPython.display import Image, display

USUARIO_GITHUB = 'gnavidad1978'
RUTA_DRIVE = '/content/drive/MyDrive/proyecto_IA'

MODELOS = {
    'mobilenet_v3_large': 'configs/mobilenet_v3.yaml',
    'yolo11n_cls': 'configs/yolo11n_cls.yaml',
    'resnet50': 'configs/resnet50.yaml',
    'vit_b_16': 'configs/vit_b_16.yaml',
}
SEMILLAS = [42, 7, 2026]

Conecto mi Google Drive, donde guardo las fotos y todos los resultados, para que no se pierdan al cerrar Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Bajo mi repositorio desde GitHub. La última línea muestra la versión exacta del código con la que obtuve estos resultados; así cualquiera puede repetir el experimento con el mismo código.

In [ ]:
%cd /content
if not os.path.exists('clasificacion_residuos_ia'):
    !git clone https://github.com/{USUARIO_GITHUB}/clasificacion_residuos_ia.git
else:
    !git -C clasificacion_residuos_ia pull
%cd /content/clasificacion_residuos_ia
!git log --oneline -1

Instalo las librerías con sus versiones fijas, para que el cuaderno dé los mismos resultados aunque salgan versiones nuevas. Además, dejo que los resultados se guarden directamente en mi Drive.

In [ ]:
!pip install -q -r requirements.txt
import torch, torchvision, lightning
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| Lightning', lightning.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

os.makedirs(f'{RUTA_DRIVE}/repo_resultados', exist_ok=True)
os.makedirs(f'{RUTA_DRIVE}/repo_runs', exist_ok=True)
if not os.path.islink('resultados'):
    !rm -rf resultados
    !ln -s "{RUTA_DRIVE}/repo_resultados" resultados
if not os.path.exists('runs'):
    !cp -r "{RUTA_DRIVE}/repo_runs" runs

## 4. Los datos

Como mi propio conjunto de fotos del campus todavía está en construcción, para este proyecto usé **RealWaste**, un conjunto público de 4,752 fotografías de residuos reales recolectados en un relleno sanitario de Australia [1]. Lo elegí porque sus residuos están sucios, aplastados y deformados, como la basura de verdad, y no son productos limpios sobre una cartulina blanca. Además trae residuos orgánicos, todas sus fotos tienen el mismo tamaño (524 por 524 píxeles) y su licencia permite usarlo citándolo.

RealWaste tiene nueve clases, así que las acomodé en mis seis categorías de la norma:

| Clase en RealWaste | Mi categoría |
|---|---|
| Cardboard y Paper | Papel y cartón |
| Glass | Vidrio |
| Metal | Metal |
| Plastic | Plástico |
| Food Organics y Vegetation | Orgánico |
| Miscellaneous Trash | Difícil reciclaje |
| Textile Trash | La excluí: en el campus no habrá bote de textiles |

Me quedaron 4,434 fotos. El acomodo de Miscellaneous Trash a difícil reciclaje es una interpretación mía que todavía estoy revisando foto por foto.

### Las reglas que seguí para que el experimento sea confiable

Antes de entrenar seguí cuatro reglas del protocolo experimental del curso.

**1. Semilla fija.** Todo lo que se hace "al azar" (cómo se reparten las fotos, en qué orden se muestran) depende de una semilla. Al fijarla, cualquiera que corra este cuaderno obtiene exactamente lo mismo.

**2. Separar los datos antes de mirarlos.** Antes de ver cualquier foto, las separé en tres grupos: 70 % para entrenar, 15 % para validar y 15 % para la prueba final. Es como un examen: si el modelo estudia con las mismas preguntas que después le van a calificar, la calificación no dice nada. El grupo de **validación** lo uso para decidir cuándo parar y qué versión del modelo quedarme; el de **prueba** lo guardé sin tocar para el final.

Además cuidé un detalle: si dos fotos son casi idénticas, las mantuve siempre en el mismo grupo. Para detectarlas calculé una especie de "huella digital" de cada foto y comparé las huellas. Así el modelo no puede "reconocer" en la validación una foto que ya vio al entrenar.

**3. Comparar contra una línea base.** El modelo más tonto posible es uno que siempre contesta la categoría más común. Si mis modelos no le ganan por mucho, no aprendieron nada.

**4. Elegir la métrica antes de ver resultados.** Lo explico en la siguiente sección.

In [ ]:
!python prepare_data.py --zip "{RUTA_DRIVE}/realwaste.zip"
display(Image('resultados/eda/conteo_entrenamiento.png'))
display(Image('resultados/eda/muestras_entrenamiento.png', width=600))

### Lo que encontré al explorar los datos

Todas las fotos de entrenamiento son JPEG, a color y del mismo tamaño, así que no tuve que corregir formatos mezclados. Las categorías **no tienen la misma cantidad de fotos**: la más grande tiene más del doble que la más chica (vidrio es la que menos tiene). Eso influyó en cómo elegí la métrica.

La línea base, que siempre contesta "papel y cartón", obtuvo una **exactitud de 0.217** pero un **F1 macro de 0.059**.

## 5. Cómo medí el desempeño

Si solo midiera la **exactitud** (cuántas fotos acierta en total), un modelo podría verse bien aunque ignore las categorías con pocas fotos. Por eso elegí como métrica principal el **F1 macro**: calcula qué tan bien le va al modelo en cada categoría por separado y después las promedia, de modo que vidrio pesa lo mismo que papel. En el bote, equivocarse con vidrio es tan grave como equivocarse con plástico.

La línea base lo deja muy claro: con exactitud parece que acierta más que el azar (0.217 contra 1/6 = 0.167), pero su F1 macro de 0.059 muestra que en realidad no sabe nada. Cualquier modelo mío tiene que estar muy por encima de 0.059.

También reviso la **matriz de confusión**, una tabla que dice qué categoría se confundió con cuál.

## 6. El entrenamiento

Entrenar es mostrarle al modelo las fotos de entrenamiento una y otra vez. Cada vuelta completa se llama **época**. En cada época el modelo comete errores, mide qué tan equivocado está (la **pérdida**) y ajusta sus pesos para equivocarse menos. Al final de cada época lo califico con las fotos de validación, que nunca usa para aprender.

Configuré estas reglas en el YAML de cada modelo:

| Parámetro | Valor | Por qué |
|---|---|---|
| Épocas máximas | 15 | Suficientes para que estos modelos se estabilicen |
| Paro temprano | Si el F1 de validación no mejora en 4 épocas | Para no seguir entrenando cuando ya no aprende y empieza a memorizar |
| Mejor modelo | Me quedo con la época de mejor F1 en validación, no con la última | Es la versión que mejor generaliza |
| Tamaño de imagen | 224 por 224 píxeles | Es el tamaño con el que se preentrenaron estas redes |
| Tasa de aprendizaje | 0.0003 en las CNN, 0.00003 en el ViT | Los transformers se desestabilizan con tasas altas |
| Aumento de datos | Volteo horizontal, giro de hasta 15° y cambios de brillo, contraste y saturación, solo en entrenamiento | Para que el modelo tolere distintas luces y posiciones |

No usé volteo vertical, porque en el bote nadie va a sostener su residuo de cabeza. Y para que la comparación fuera justa, entrené los cuatro modelos con exactamente las mismas fotos, los mismos aumentos y las mismas reglas; solo cambia la red. Por ejemplo, el archivo `configs/vit_b_16.yaml` contiene, entre otras, estas líneas:

```yaml
trainer:
  max_epochs: 15
model:
  backbone: vit_b_16
  lr: 0.00003
data:
  img_size: 224
```

Para observar el entrenamiento usé **TensorBoard**, que dibuja las curvas de pérdida y de F1 de cada corrida.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Esta celda entrena los cuatro modelos con las tres semillas, es decir, doce entrenamientos. Cada uno es un solo comando, por ejemplo `python main.py fit --config configs/vit_b_16.yaml --seed_everything 7`. Si un entrenamiento ya está guardado en mi Drive, lo salta, así que puedo continuar donde me quedé.

In [ ]:
for backbone, config in MODELOS.items():
    for semilla in SEMILLAS:
        if os.path.exists(f'resultados/{backbone}_semilla{semilla}.json'):
            print(f'{backbone}, semilla {semilla}: ya entrenado')
            continue
        comando = f'python main.py fit --config {config} --seed_everything {semilla}'
        !{comando}
        respaldo = f'cp -r runs/. "{RUTA_DRIVE}/repo_runs/"'
        !{respaldo}

## 7. Cuánto cuesta cada modelo

Al principio comparé el **tiempo total de entrenamiento**, pero descubrí que no servía: los cuatro modelos tardaron casi lo mismo, unos 8 minutos, aunque el ViT tiene 57 veces más parámetros que YOLO. Incluso por época, en la GPU, las tres CNN tardaron entre 33 y 38 segundos y el ViT 53. Había dos razones. La primera, que en Colab gratuito solo hay dos procesadores para abrir y preparar las fotos, y ese paso tarda lo mismo para cualquier red; la GPU pasaba buena parte del tiempo esperando las fotos. La segunda, que cada modelo se detuvo en una época distinta.

Para medir el costo real de cada red, en esta celda le doy a cada modelo datos ya cargados en memoria y cuento **cuántas fotos por segundo puede procesar**, al entrenar y al clasificar. Así solo cuenta el cómputo de la arquitectura. Hice esta medición en el procesador de Colab (dos núcleos, sin GPU), que además se parece más a la computadora pequeña que irá en el bote. Como tarda alrededor de 45 minutos, la celda solo la vuelve a correr si no encuentra el resultado guardado.

Además, durante el entrenamiento medí el tiempo de clasificar **una foto a la vez en el procesador**, con 4 hilos, como aproximación a la Raspberry Pi 5, que tiene 4 núcleos. No sustituye medir en la Raspberry Pi real; eso lo haré en mi tesis.

In [ ]:
if not os.path.exists('resultados/costo_computo.csv'):
    !python benchmark.py
else:
    import pandas as pd
    print(pd.read_csv('resultados/costo_computo.csv', index_col='modelo').round(1).to_string())

| Modelo | Fotos por segundo al entrenar | Fotos por segundo al clasificar |
|---|---|---|
| YOLO11n-cls | 22.4 | 73.0 |
| MobileNetV3-Large | 7.4 | 32.5 |
| ResNet-50 | 1.4 | 4.8 |
| ViT-B/16 | 0.5 | 1.7 |

Aquí sí aparece la diferencia que el tiempo total escondía. Con el cómputo aislado, el ViT es unas 15 veces más lento que MobileNetV3 para entrenar y unas 19 veces más lento para clasificar; frente a YOLO la diferencia es de más de 40 veces. ResNet-50 clasifica casi 7 veces más lento que MobileNetV3. El orden coincide con el número de parámetros de cada red.

## 8. Resultados

In [ ]:
!python report.py

In [ ]:
for nombre in ['curvas', 'confusion_mobilenet_v3_large', 'confusion_resnet50', 'confusion_vit_b_16',
               'confusion_yolo11n_cls', 'errores_mobilenet_v3_large', 'errores_vit_b_16']:
    ruta = f'resultados/figuras/{nombre}.png'
    if os.path.exists(ruta):
        display(Image(ruta, width=1000 if nombre in ('curvas',) or nombre.startswith('errores') else 430))

### Lo que obtuve

Estos son los resultados en el grupo de validación, como promedio ± desviación estándar de las tres semillas:

| Modelo | Familia | F1 macro | Parámetros | Tamaño | Tiempo por foto en CPU |
|---|---|---|---|---|---|
| ResNet-50 | CNN | 0.962 ± 0.008 | 23.5 millones | 94 MB | 160 ms |
| MobileNetV3-Large | CNN | 0.957 ± 0.004 | 4.2 millones | 17 MB | 46 ms |
| ViT-B/16 | Transformer | 0.939 ± 0.008 | 85.8 millones | 343 MB | 532 ms |
| YOLO11n-cls | CNN | 0.913 ± 0.003 | 1.5 millones | 6 MB | 24 ms |
| Línea base | | 0.059 | | | |

Todos los modelos aprendieron: están muy por encima de la línea base de 0.059. Para decidir si dos modelos son realmente distintos uso una regla sencilla: la diferencia entre ellos tiene que ser claramente mayor que su desviación estándar.

**El Vision Transformer no ganó; quedó por debajo.** Obtuvo unos dos puntos menos que ResNet-50 y MobileNetV3, una diferencia de entre dos y tres veces su desviación, así que difícilmente es suerte. Y es, por mucho, el más costoso: 20 veces más parámetros que MobileNetV3 y más de 11 veces más lento para clasificar una foto.

**ResNet-50 y MobileNetV3 empataron.** Su diferencia (0.005) es menor que su variación entre semillas. Con el mismo desempeño, MobileNetV3 es casi seis veces más pequeña y más de tres veces más rápida. Es el mejor balance entre costo y desempeño, y la candidata natural para la Raspberry Pi.

**YOLO11n-cls es el más ligero y el más rápido, pero también el más bajo**, unos cuatro a cinco puntos debajo. Su resultado fue muy estable entre semillas.

### Lo que muestran las curvas

En los cuatro modelos la pérdida de entrenamiento llega casi a cero mientras la de validación se estanca: el modelo empieza a memorizar las fotos de entrenamiento. El paro temprano lo detectó y me quedé con la mejor época. El ViT fue el que sobreajustó más rápido: en promedio se detuvo a las 9 épocas, porque su mejor resultado llegó pronto y después la pérdida de validación empezó a subir. MobileNetV3 y ResNet-50 entrenaron entre 13 y 14 épocas. YOLO, en cambio, llegó al tope de 15 épocas en las tres semillas, es decir, seguía mejorando de vez en cuando; es posible que con más épocas alcance un mejor resultado. Además, su pérdida de validación quedó más alta que la de los demás.

### Dónde se equivocan

Este es el F1 de cada categoría, promedio de las tres semillas:

| Categoría | MobileNetV3 | ResNet-50 | ViT-B/16 | YOLO11n-cls |
|---|---|---|---|---|
| Vidrio | 0.979 | 0.977 | 0.961 | 0.961 |
| Plástico | 0.945 | 0.949 | 0.936 | 0.893 |
| Metal | 0.941 | 0.939 | 0.924 | 0.884 |
| Papel y cartón | 0.971 | 0.980 | 0.958 | 0.930 |
| Difícil reciclaje | **0.922** | **0.938** | **0.877** | **0.844** |
| Orgánico | 0.984 | 0.990 | 0.980 | 0.966 |

Los cuatro modelos coinciden: la categoría más difícil es **difícil reciclaje** (en ResNet-50 prácticamente empatada con metal) y la segunda es **metal**. Orgánico y vidrio son las más fáciles. Si los cuatro modelos fallan en el mismo lugar, el problema no es la arquitectura sino la categoría. Tiene sentido, porque difícil reciclaje proviene de "Miscellaneous Trash", la clase más revuelta de RealWaste.

El otro error frecuente fue confundir **metal con plástico**. Al revisar las fotos mal clasificadas encontré aerosoles con su tapa de plástico a la vista, latas con etiquetas impresas y papel aluminio arrugado que el modelo confunde con papel. El modelo se guía por lo que ve en la superficie y no por el material de abajo, algo que coincide con lo que vimos en clase sobre el sesgo de las CNN hacia la textura [6].

También encontré fotos donde la etiqueta original de RealWaste no coincide con la norma de Jalisco. Por ejemplo, la norma pone el papel aluminio, las envolturas metalizadas y los envases multicapa tipo Tetra Pak en "difícil reciclaje", mientras que RealWaste los etiqueta como metal, plástico o cartón. En esos casos el modelo "se equivoca" respecto a RealWaste, pero la etiqueta de RealWaste tampoco es la correcta para Jalisco.

## 9. Conclusiones

Respondiendo a mi pregunta: **para este problema, el Vision Transformer no justifica su costo.** No solo es el modelo más pesado y más lento; también obtuvo menor desempeño que las dos mejores redes convolucionales. MobileNetV3 alcanzó el mismo desempeño que ResNet-50 con una fracción del tamaño y del tiempo, por lo que es la mejor opción para un bote que funcionará con energía solar.

### Limitaciones

Es importante decir lo que estos resultados **no** demuestran.

Primero, RealWaste no tiene residuos sostenidos en la mano, que es como la gente usará el bote; los residuos aparecen solos sobre un piso de concreto. Por eso espero que el desempeño baje cuando use fotos reales del campus.

Segundo, entrené los cuatro modelos con la misma receta para que la comparación fuera justa, pero no ajusté los parámetros de cada uno por separado. YOLO, en particular, se diseñó con otra receta de entrenamiento y llegó al tope de épocas sin detenerse, así que es posible que con más épocas y ajustes propios mejore.

Tercero, las tres semillas miden cuánto varía el entrenamiento, pero las tres usan la misma separación de las fotos.

Cuarto, RealWaste no indica qué fotos son del mismo objeto tomado desde otro ángulo, así que algunas podrían haber quedado en grupos distintos y hacer que los resultados de validación se vean un poco mejores de lo que son.

### Qué sigue

Me falta evaluar una sola vez con el grupo de prueba, que mantuve guardado durante todo el proyecto, y probar el mejor modelo con fotos tomadas en el campus de residuos sostenidos en la mano. En esas fotos el residuo ocupa una parte pequeña de la imagen, así que primero usaré un detector YOLO para recortarlo y después lo clasificaré con la CNN. Esa diferencia entre laboratorio y campo es justamente lo que mi tesis busca medir.

## 10. Evaluación final con el grupo de prueba

Esta evaluación la reservé para el final y la hago **una sola vez**, cuando ya no voy a cambiar modelos ni parámetros. Uso la mejor versión guardada de cada modelo.

In [ ]:
def evaluar_prueba(backbone, semilla):
    r = json.load(open(f'resultados/{backbone}_semilla{semilla}.json'))
    comando = f'python main.py test --config {MODELOS[backbone]} --ckpt_path "{r["checkpoint"]}"'
    get_ipython().system(comando)

# Se ejecuta una sola vez al final del proyecto:
# for b in MODELOS: evaluar_prueba(b, 42)

## Referencias

[1] S. Single, S. Iranmanesh y R. Raad, "RealWaste: A Novel Real-Life Data Set for Landfill Waste Classification Using Deep Learning," *Information*, vol. 14, no. 12, p. 633, 2023, doi: 10.3390/info14120633. Conjunto de datos en UCI Machine Learning Repository, doi: 10.24432/C5SS4G.

[2] K. He, X. Zhang, S. Ren y J. Sun, "Deep Residual Learning for Image Recognition," en *Proc. IEEE Conf. Computer Vision and Pattern Recognition (CVPR)*, 2016, pp. 770–778.

[3] A. Howard *et al.*, "Searching for MobileNetV3," en *Proc. IEEE/CVF Int. Conf. Computer Vision (ICCV)*, 2019, pp. 1314–1324.

[4] A. Dosovitskiy *et al.*, "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale," en *Proc. Int. Conf. Learning Representations (ICLR)*, 2021.

[5] G. Jocher y J. Qiu, *Ultralytics YOLO11*, versión 11.0.0, 2024. [En línea]. Disponible: https://github.com/ultralytics/ultralytics

[6] R. Geirhos, P. Rubisch, C. Michaelis, M. Bethge, F. A. Wichmann y W. Brendel, "ImageNet-trained CNNs are biased towards texture; increasing shape bias improves accuracy and robustness," en *Proc. Int. Conf. Learning Representations (ICLR)*, 2019.

[7] J. Deng, W. Dong, R. Socher, L.-J. Li, K. Li y L. Fei-Fei, "ImageNet: A Large-Scale Hierarchical Image Database," en *Proc. IEEE Conf. Computer Vision and Pattern Recognition (CVPR)*, 2009, pp. 248–255.

[8] W. Falcon y The PyTorch Lightning team, *PyTorch Lightning*, 2019, doi: 10.5281/zenodo.3828935.

[9] Secretaría de Medio Ambiente para el Desarrollo Sustentable, "Norma Ambiental Estatal NAE-SEMADES-007/2008," *Periódico Oficial El Estado de Jalisco*, núm. 35, secc. III, 16 de octubre de 2008.